# 1. Dati e analisi descrittive

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedeGambe/Master_s_thesis_Data_science/blob/main/notebooks/01_dati_e_analisi_descrittive.ipynb)

Caricamento del dataset, controlli di qualità, distribuzioni univariate e relazione di ogni variabile con il possesso di un BEV (capitoli 1-3 della tesi).

In [ ]:
# Configurazione dell'ambiente (locale o Google Colab)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("Master_s_thesis_Data_science"):
        !git clone -q -b main https://github.com/FedeGambe/Master_s_thesis_Data_science.git
    %cd Master_s_thesis_Data_science
    !pip install -q -r requirements-colab.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import plotly.io as pio
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)
if os.environ.get("TESI_GRAFICI_STATICI"):
    pio.renderers.default = "png"  # usato solo per salvare output visibili su GitHub
    pio.defaults.default_width = 1000

In [ ]:
from tesi_bev import config as C, dati, statistica as S, grafici as G
df = dati.carica_dati()
print(f"{df.shape[0]} osservazioni, {df.shape[1]} variabili, {df.isna().sum().sum()} valori mancanti, "
      f"{df.duplicated().sum()} duplicati")
df.head()

## 1.1 Popolazione e variabile dipendente

Tutti i rispondenti possiedono già un veicolo a basse emissioni (BEV, PHEV, FCEV o HEV): il modello quindi non
stima la probabilità di comprare *un'auto elettrica* in generale, ma la probabilità che il veicolo posseduto sia un
**BEV** rispetto alle altre alimentazioni alternative.

In [ ]:
pd.concat([S.frequenze(df, C.TIPO_AUTO_ATTUALE), S.frequenze(df, C.TARGET)], keys=["Auto attuale", "BEV dummy"])

## 1.2 Variabili quantitative

In [ ]:
S.descrittive_numeriche(df)

In [ ]:
for col in [C.REDDITO, C.EMISSIONI, C.VMT, C.N_VIAGGI_LUNGHI]:
    G.distribuzione(df, col, log_x=col in (C.VMT,)).show()

## 1.3 Quota di BEV per categoria

In [ ]:
for col, ordine in [(C.CLASSE_REDDITO, C.ORDINE_REDDITO), (C.CLASSE_ETA, C.ORDINE_ETA),
                    (C.ISTRUZIONE, C.ORDINE_ISTRUZIONE), (C.AUTO_PRECEDENTE, C.TIPI_AUTO)]:
    G.quota_bev(df, col, ordine).show()

## 1.4 Test bivariati

Chi-quadro con V di Cramér per le variabili categoriali, Mann-Whitney con correlazione rank-biserial per quelle
numeriche (le distribuzioni sono fortemente asimmetriche, quindi un test non parametrico è più adatto del t-test).
Con oltre 10.000 osservazioni quasi tutto risulta significativo: conviene guardare la **dimensione dell'effetto**.

In [ ]:
S.test_bivariati(df).sort_values('effetto', key=abs, ascending=False)

## 1.5 Correlazioni tra le feature dei modelli

In [ ]:
X, y = dati.prepara_ml(df)
G.matrice_correlazione(X).show()